In [4]:
import urllib.request
import zipfile

url = "https://storage.googleapis.com/laurencemoroney-blog.appspot.com/horse-or-human.zip"

file_name = "horse-or-human.zip"
training_dir = "horse-or-human/training"
urllib.request.urlretrieve(url, file_name)

zip_ref = zipfile.ZipFile(file_name, 'r')
zip_ref.extractall(training_dir)
zip_ref.close()

HTTPError: HTTP Error 404: Not Found

In [5]:
# python
import requests
import zipfile
import shutil
from pathlib import Path
import tempfile
import os

URL = "https://storage.googleapis.com/laurencemoroney-blog.appspot.com/horse-or-human.zip"
target_dir = Path("horse-or-human") / "training"
target_dir.mkdir(parents=True, exist_ok=True)

# download to a temporary file (streaming)
with requests.get(URL, stream=True) as r:
    r.raise_for_status()
    with tempfile.NamedTemporaryFile(delete=False) as tmp:
        for chunk in r.iter_content(chunk_size=8192):
            if chunk:
                tmp.write(chunk)
        tmp_path = Path(tmp.name)

# safe extraction to avoid Zip Slip
def safe_extract(zip_path: Path, dest: Path):
    with zipfile.ZipFile(zip_path, "r") as z:
        for member in z.infolist():
            member_path = dest / member.filename
            # Resolve to absolute paths and ensure the target is inside dest
            if not Path(os.path.abspath(member_path)).resolve().is_relative_to(Path(os.path.abspath(dest)).resolve()):
                raise Exception(f"Unsafe zip entry: {member.filename}")
        z.extractall(dest)

try:
    safe_extract(tmp_path, target_dir)
finally:
    # cleanup temp file
    try:
        tmp_path.unlink()
    except Exception:
        pass


HTTPError: 404 Client Error: Not Found for url: https://storage.googleapis.com/laurencemoroney-blog.appspot.com/horse-or-human.zip